# 365 Probabilidades - Dia #136
## Qual a probabilidade de nascerem mais bebês na lua cheia?

**Tipo:** Mito testado com dado
**Data de publicação:** 2026-10-27
**Ferramenta:** Python
**Decisão analisada:** A lua cheia lota a maternidade, ou é o calendário do hospital que manda?
**Hashtag:** #365Probabilidades #Dia136

---

### 📖 A História

Esses dias, conversando com uma tia, ouvi a história de uma conhecida dela que estava
grávida. A mala da maternidade já estava pronta, porque quando a lua virasse o bebê ia
nascer. Disso ela tinha certeza.

Guardei o tema e fiquei esperando a lua virar, para ver o que aconteceria. Enquanto
isso, fui atrás dos dados.

Quem já trabalhou numa maternidade costuma ter uma opinião sobre o que a lua cheia faz
com o plantão. Quem já pariu também. É uma das crenças mais antigas sobre o corpo, e uma
das mais fáceis de testar, porque o Brasil registra o dia de nascimento de todos os bebês.

A lua ficou cheia ontem, à 1h12 da manhã. Fui contar, até que o bebê de verdade
nascesse, ou não.

---

### 📚 O Conceito: o que o olho vê e o que a conta vê

Um plantão cheio numa noite de lua cheia fica na memória. Um plantão cheio numa noite
qualquer, não. A psicologia chama isso de correlação ilusória: a gente guarda as
coincidências que confirmam a ideia e esquece as que não confirmam.

A estatística tem um jeito de contornar a memória. Pega todos os dias, não só os
marcantes, e compara cada um com o que seria esperado para ele. Esperado para uma
terça-feira de março, para um domingo de julho, para o Natal. Se a lua cheia faz algo,
ela aparece como um excesso sobre esse esperado. Se não faz, os dias de lua cheia ficam
espalhados em volta do esperado como qualquer outro dia.

Quem fez isso com mais dados foi uma dupla da Universidade de Halle, na Alemanha, com
4 milhões de nascimentos em 38 anos. Não encontrou nada. Este notebook refaz a conta com
15,7 milhões de nascimentos brasileiros.

---

### 🧮 O Modelo

Cinco camadas. Os dados são do SINASC, o registro nacional de nascidos vivos. As contas
são nossas.

1. **O calendário do hospital.** Quanto o dia da semana e o feriado mudam o número de
   nascimentos. É a régua para medir qualquer outro efeito.
2. **As 74 luas cheias.** Para cada dia de lua cheia, o número de nascimentos dividido
   pelo esperado para aquele dia. A razão agregada, com intervalo de 95% por reamostragem,
   e a contagem de luas cheias acima do esperado, com intervalo de Jeffreys.
3. **O calendário deslocado.** O mesmo cálculo com o calendário lunar adiantado ou
   atrasado de 1 a 15 dias. Trinta calendários falsos para situar o verdadeiro.
4. **Só os partos vaginais.** A cesárea tem dia marcado, e o dia marcado pode esconder a
   lua. A camada repete a conta só nos partos em que o hospital não escolhe a data.
5. **O periodograma.** O método do estudo alemão: a série diária é decomposta em ciclos e
   a pergunta é se existe um ciclo de 29,53 dias, que é o período da lua.

**Fonte central:**
- Kuss, O. & Kuehn, A. (2008). "Lunar cycle and the number of births: a spectral
  analysis of 4,071,669 births from South-Western Germany". *Acta Obstetricia et
  Gynecologica Scandinavica* 87(12), 1378-1379. DOI 10.1080/00016340802478174.
  **N = 4.071.669** nascimentos (vivos e mortos) em Baden-Württemberg, de janeiro de 1966
  a dezembro de 2003. Análise espectral da série diária, com tendência removida por
  spline cúbica e teste F na ordenada do período sinódico de 29,53 dias: **p = 0,688**.
  Os picos do periodograma estão no ciclo semanal e no anual. Publicado como carta ao
  editor, sem resumo.

**Fonte de apoio:**
- Arliss, J. M., Kaplan, E. N. & Galvin, S. L. (2005). "The effect of the lunar cycle on
  frequency of births and birth complications". *American Journal of Obstetrics and
  Gynecology* 192(5), 1462-1464. N = 564.039 nascimentos na Carolina do Norte, de 1997 a
  2001, em 62 ciclos lunares. Sem diferença entre as oito fases da lua em frequência de
  nascimentos, via de parto ou complicações.

**Dados:**
- SINASC / Ministério da Saúde, microdados por ano (OpenDataSUS), 2019 a 2024. Contagem
  própria de nascimentos por dia de nascimento (campo DTNASC) e por tipo de parto (campo
  PARTO), feita sobre os arquivos anuais completos. **N = 15.745.215** nascidos vivos em
  2.192 dias. Arquivo `dia-136-sinasc-nascimentos-por-dia.csv`.
- Fases da lua calculadas pelo algoritmo de Meeus (*Astronomical Algorithms*, cap. 49),
  no horário de Brasília. Conferido com as efemérides: lua cheia de 26/10/2026 à 1h12.

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. Nenhum número aqui é
autorrelato: são registros administrativos de nascimento e posições da lua. O dia de lua
cheia é o dia civil, em Brasília, que contém o instante da lua cheia.

In [ ]:
import math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from dateutil.easter import easter

SEMENTE = 42
REAMOSTRAS = 20_000
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = ('Fonte: SINASC/Ministério da Saúde 2019-2024, 15,7 milhões de nascimentos · fases da lua por Meeus · '
          'cálculo próprio | #365Probabilidades')

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def jeffreys(k, n):
    return stats.beta(k + 0.5, n - k + 0.5)

def abre(arquivo):
    for pasta in ["", os.path.join("modelos", "dias-121-150"), os.path.join("..", "dias-121-150")]:
        caminho = os.path.join(pasta, arquivo)
        if os.path.exists(caminho):
            return pd.read_csv(caminho)
    raise FileNotFoundError(arquivo + " precisa estar na mesma pasta do notebook")

# ---- Fases da lua · Meeus, Astronomical Algorithms, cap. 49 ----
SINODICO = 29.530588861           # dias · período médio entre duas luas novas

def _fase_jde(k, fase):
    # instante (dia juliano) da fase · fase: 0 nova, 0.5 cheia
    k = k + fase
    T = k / 1236.85
    jde = 2451550.09766 + SINODICO * k + 0.00015437 * T**2 - 1.5e-7 * T**3 + 7.3e-10 * T**4
    E = 1 - 0.002516 * T - 0.0000074 * T**2
    M  = math.radians((2.5534 + 29.10535670 * k - 0.0000014 * T**2 - 1.1e-7 * T**3) % 360)
    Ml = math.radians((201.5643 + 385.81693528 * k + 0.0107582 * T**2 + 0.00001238 * T**3 - 5.8e-8 * T**4) % 360)
    F  = math.radians((160.7108 + 390.67050284 * k - 0.0016118 * T**2 - 0.00000227 * T**3 + 1.1e-8 * T**4) % 360)
    Om = math.radians((124.7746 - 1.56375588 * k + 0.0020672 * T**2 + 0.00000215 * T**3) % 360)
    s = math.sin
    c = ([-0.40720, 0.17241, 0.01608, 0.01039, 0.00739, -0.00514, 0.00208] if fase == 0 else
         [-0.40614, 0.17302, 0.01614, 0.01043, 0.00734, -0.00515, 0.00209])
    corr = (c[0]*s(Ml) + c[1]*E*s(M) + c[2]*s(2*Ml) + c[3]*s(2*F) + c[4]*E*s(Ml-M) + c[5]*E*s(Ml+M)
            + c[6]*E*E*s(2*M) - 0.00111*s(Ml-2*F) - 0.00057*s(Ml+2*F) + 0.00056*E*s(2*Ml+M)
            - 0.00042*s(3*Ml) + 0.00042*E*s(M+2*F) + 0.00038*E*s(M-2*F) - 0.00024*E*s(2*Ml-M)
            - 0.00017*s(Om) - 0.00007*s(Ml+2*M) + 0.00004*s(2*Ml-2*F) + 0.00004*s(3*M)
            + 0.00003*s(Ml+M-2*F) + 0.00003*s(2*Ml+2*F) - 0.00003*s(Ml+M+2*F) + 0.00003*s(Ml-M+2*F)
            - 0.00002*s(Ml-M-2*F) - 0.00002*s(3*Ml+M) + 0.00002*s(4*Ml))
    A = [299.77 + 0.107408*k - 0.009173*T**2, 251.88 + 0.016321*k, 251.83 + 26.651886*k,
         349.42 + 36.412478*k, 84.66 + 18.206239*k, 141.74 + 53.303771*k, 207.14 + 2.453732*k,
         154.84 + 7.306860*k, 34.52 + 27.261239*k, 207.19 + 0.121824*k, 291.34 + 1.844379*k,
         161.72 + 24.198154*k, 239.56 + 25.513099*k, 331.55 + 3.592518*k]
    ca = [0.000325, 0.000165, 0.000164, 0.000126, 0.000110, 0.000062, 0.000060,
          0.000056, 0.000047, 0.000042, 0.000040, 0.000037, 0.000035, 0.000023]
    corr += sum(ci * math.sin(math.radians(a % 360)) for ci, a in zip(ca, A))
    return jde + corr

def fases_da_lua(ano_ini, ano_fim, fuso='America/Sao_Paulo'):
    # tabela de luas novas e cheias entre os anos, no fuso pedido
    k0 = math.floor((ano_ini - 2000) * 12.3685) - 2
    k1 = math.ceil((ano_fim + 1 - 2000) * 12.3685) + 2
    linhas = []
    for k in range(k0, k1):
        for f, nome in ((0, 'nova'), (0.5, 'cheia')):
            jde = _fase_jde(k, f)
            t = pd.Timestamp('2000-01-01 12:00', tz='UTC') + pd.to_timedelta(jde - 2451545.0, unit='D')
            linhas.append({'fase': nome, 'instante': t.tz_convert(fuso).tz_localize(None)})
    df = pd.DataFrame(linhas).sort_values('instante').reset_index(drop=True)
    return df[(df.instante.dt.year >= ano_ini) & (df.instante.dt.year <= ano_fim)].reset_index(drop=True)

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS ---
d = abre("dia-136-sinasc-nascimentos-por-dia.csv")
d['data'] = pd.to_datetime(d['data'])
d = d.sort_values('data').reset_index(drop=True)

# Lua: dia civil (Brasília) que contém o instante da lua cheia
fases = fases_da_lua(2018, 2026)
cheias = fases[fases.fase == 'cheia'].instante
d['cheia'] = d.data.isin(set(cheias.dt.normalize())).astype(int)

# Feriados nacionais e vésperas que esvaziam o hospital
feriados = set()
for ano in range(2019, 2025):
    pascoa = pd.Timestamp(easter(ano))
    for dias in (-48, -47, -46, -2, 60):          # carnaval (seg, ter, cinzas), sexta santa, Corpus Christi
        feriados.add(pascoa + pd.Timedelta(days=dias))
    for md in ['01-01', '04-21', '05-01', '09-07', '10-12', '11-02', '11-15', '12-24', '12-25', '12-31']:
        feriados.add(pd.Timestamp(f'{ano}-{md}'))
feriados.add(pd.Timestamp('2024-11-20'))         # Consciência Negra, nacional a partir de 2024
d['feriado'] = d.data.isin(feriados).astype(int)
d['dow'] = d.data.dt.dayofweek
d['ano_mes'] = d.data.dt.strftime('%Y-%m')

DIAS_SEMANA = ['seg', 'ter', 'qua', 'qui', 'sex', 'sáb', 'dom']
N_TOTAL = int(d.nascimentos.sum())

print("=" * 70)
print("  DADOS · SINASC 2019-2024, NASCIMENTOS POR DIA")
print("=" * 70)
print(f"\n  {mil(N_TOTAL)} nascidos vivos em {mil(len(d))} dias ({d.data.min():%d/%m/%Y} a {d.data.max():%d/%m/%Y})")
print(f"  Média diária: {mil(d.nascimentos.mean())} · mínimo {mil(d.nascimentos.min())} ({d.loc[d.nascimentos.idxmin(), 'data']:%d/%m/%Y}) "
      f"· máximo {mil(d.nascimentos.max())} ({d.loc[d.nascimentos.idxmax(), 'data']:%d/%m/%Y})")
print(f"  Partos vaginais: {vir(d.vaginal.sum() / N_TOTAL * 100)}% · cesáreas: {vir(d.cesarea.sum() / N_TOTAL * 100)}% "
      f"· tipo ignorado: {vir(d.ignorado.sum() / N_TOTAL * 100, 2)}%")
print(f"\n  Dias de lua cheia no período: {d.cheia.sum()} · feriados e vésperas marcados: {d.feriado.sum()}")
print("\n  Por ano:")
for ano, g in d.groupby(d.data.dt.year):
    print(f"    {ano}  {mil(g.nascimentos.sum()):>10}  ({len(g)} dias · {g.cheia.sum()} luas cheias)")
print("\n  Média bruta por dia da semana:")
for i, nome in enumerate(DIAS_SEMANA):
    print(f"    {nome}  {mil(d[d.dow == i].nascimentos.mean()):>6}")
print(f"\n  Próximas luas cheias: " + " · ".join(f"{t:%d/%m/%Y %Hh%M}" for t in cheias[cheias >= '2026-10-01'].head(3)))
print("=" * 70)

In [ ]:
# --- O MODELO ---

# Regressão de Poisson por mínimos quadrados reponderados (IRLS), com escala quase-Poisson.
# log E[nascimentos] = dia da semana + mês do ano + feriado  (+ lua cheia, quando testada)
def poisson_irls(X, y):
    b = np.linalg.lstsq(X, np.log(y), rcond=None)[0]
    for _ in range(50):
        mu = np.exp(X @ b)
        z = X @ b + (y - mu) / mu
        XtW = X.T * mu
        b_novo = np.linalg.solve(XtW @ X, XtW @ z)
        if np.max(np.abs(b_novo - b)) < 1e-10:
            b = b_novo
            break
        b = b_novo
    mu = np.exp(X @ b)
    phi = np.sum((y - mu) ** 2 / mu) / (len(y) - X.shape[1])     # dispersão quase-Poisson
    cov = np.linalg.inv((X.T * mu) @ X) * phi
    return b, mu, phi, cov

X_cal = pd.concat([pd.get_dummies(d.dow, prefix='dow', drop_first=True),
                   pd.get_dummies(d.ano_mes, prefix='m'),
                   d[['feriado']]], axis=1).astype(float).values
K_CAL = X_cal.shape[1]

def ajusta(coluna):
    y = d[coluna].values.astype(float)
    b, mu, phi, _ = poisson_irls(X_cal, y)
    b2, _, _, cov2 = poisson_irls(np.column_stack([X_cal, d.cheia.values]), y)
    se = math.sqrt(cov2[-1, -1])
    return dict(b=b, esperado=mu, phi=phi, razao_modelo=math.exp(b2[-1]),
                ic_modelo=(math.exp(b2[-1] - 1.96 * se), math.exp(b2[-1] + 1.96 * se)))

def razao_bootstrap(obs, esp):
    idx = rng.integers(0, len(obs), (REAMOSTRAS, len(obs)))
    bs = obs[idx].sum(axis=1) / esp[idx].sum(axis=1)
    return obs.sum() / esp.sum(), np.percentile(bs, [2.5, 97.5]), np.mean(bs > 1)

# CAMADA 1 · O calendário do hospital
tot = ajusta('nascimentos')
d['esperado'] = tot['esperado']
efeito_dow = np.exp(np.r_[0, tot['b'][:6]])
efeito_feriado = math.exp(tot['b'][K_CAL - 1])
print("=" * 70)
print("  CAMADA 1 · O CALENDÁRIO DO HOSPITAL")
print("=" * 70)
print("\n  Nascimentos em relação a uma segunda-feira do mesmo mês:")
for nome, e in zip(DIAS_SEMANA, efeito_dow):
    print(f"    {nome}  {vir((e - 1) * 100, 1):>6}%")
print(f"    feriado  {vir((efeito_feriado - 1) * 100, 1):>6}%")
vag = ajusta('vaginal'); ces = ajusta('cesarea')
print(f"\n  Domingo, só cesáreas:        {vir((math.exp(ces['b'][5]) - 1) * 100, 1)}%")
print(f"  Domingo, só partos vaginais: {vir((math.exp(vag['b'][5]) - 1) * 100, 1)}%")
print(f"  Dispersão quase-Poisson do modelo: {vir(tot['phi'], 1)} (o dia a dia varia mais do que uma Poisson pura)")
print(f"  Desvio típico do dia em torno do esperado: {vir(np.std(np.log(d.nascimentos / d.esperado)) * 100, 1)}%")

# CAMADA 2 · As 74 luas cheias
lc = d[d.cheia == 1].copy()
lc['razao'] = lc.nascimentos / lc.esperado
razao, ic_bs, p_maior = razao_bootstrap(lc.nascimentos.values.astype(float), lc.esperado.values)
acima = int((lc.razao > 1).sum()); n_lc = len(lc)
ic_acima = jeffreys(acima, n_lc).interval(0.95)
excesso_dia = (razao - 1) * d.nascimentos.mean()
print("\n" + "=" * 70)
print(f"  CAMADA 2 · AS {n_lc} LUAS CHEIAS")
print("=" * 70)
print(f"\n  Nascimentos observados nos dias de lua cheia: {mil(lc.nascimentos.sum())}")
print(f"  Esperado pelo calendário:                     {mil(lc.esperado.sum())}")
print(f"  Razão observado/esperado: {vir(razao, 4)}  IC 95% por reamostragem [{vir(ic_bs[0], 4)}; {vir(ic_bs[1], 4)}]")
print(f"  Pelo modelo com a lua como variável: {vir(tot['razao_modelo'], 4)}  [{vir(tot['ic_modelo'][0], 4)}; {vir(tot['ic_modelo'][1], 4)}]")
print(f"  Em bebês: {vir(excesso_dia, 0)} a mais num dia de {mil(d.nascimentos.mean())}")
print(f"  Chance de a razão ser maior que 1: {vir(p_maior * 100, 0)}%")
print(f"\n  Luas cheias com mais nascimentos do que o esperado: {acima} de {n_lc} = {vir(acima / n_lc * 100)}%")
print(f"  IC 95% (Jeffreys): [{vir(ic_acima[0] * 100)}%; {vir(ic_acima[1] * 100)}%]  ·  uma moeda honesta dá 50%")
print(f"  Lua cheia mais cheia: {lc.loc[lc.razao.idxmax(), 'data']:%d/%m/%Y} ({vir((lc.razao.max() - 1) * 100)}% acima) · "
      f"mais vazia: {lc.loc[lc.razao.idxmin(), 'data']:%d/%m/%Y} ({vir((lc.razao.min() - 1) * 100)}%)")
janela = d[(d.cheia.shift(1, fill_value=0) + d.cheia + d.cheia.shift(-1, fill_value=0)) > 0]
print(f"  Janela de três dias (véspera, dia e dia seguinte): razão {vir(janela.nascimentos.sum() / janela.esperado.sum(), 4)}")

# CAMADA 3 · O calendário deslocado
idx_cheia = np.where(d.cheia.values == 1)[0]
DESLOC = range(-15, 15)
razoes_desloc = []
for k in DESLOC:
    i = idx_cheia + k
    i = i[(i >= 0) & (i < len(d))]
    razoes_desloc.append(d.nascimentos.values[i].sum() / d.esperado.values[i].sum())
razoes_desloc = np.array(razoes_desloc)
posicao = int((razoes_desloc >= razao).sum())
print("\n" + "=" * 70)
print("  CAMADA 3 · O CALENDÁRIO DESLOCADO")
print("=" * 70)
print(f"\n  Razão observado/esperado com o calendário lunar deslocado de -15 a +14 dias:")
print(f"  mínimo {vir(razoes_desloc.min(), 4)} · mediana {vir(np.median(razoes_desloc), 4)} · máximo {vir(razoes_desloc.max(), 4)}")
print(f"  O calendário verdadeiro ({vir(razao, 4)}) fica em {posicao}º lugar entre {len(razoes_desloc)}.")
print(f"  Se a lua não fizesse nada, a chance de ficar em {posicao}º ou acima seria de {vir(posicao / len(razoes_desloc) * 100, 0)}%.")

# CAMADA 4 · Só os partos vaginais
d['esperado_vag'] = vag['esperado']
lv = d[d.cheia == 1]
razao_v, ic_v, p_maior_v = razao_bootstrap(lv.vaginal.values.astype(float), lv.esperado_vag.values)
acima_v = int((lv.vaginal / lv.esperado_vag > 1).sum())
razoes_desloc_v = []
for k in DESLOC:
    i = idx_cheia + k
    i = i[(i >= 0) & (i < len(d))]
    razoes_desloc_v.append(d.vaginal.values[i].sum() / d.esperado_vag.values[i].sum())
posicao_v = int((np.array(razoes_desloc_v) >= razao_v).sum())
print("\n" + "=" * 70)
print("  CAMADA 4 · SÓ OS PARTOS VAGINAIS")
print("=" * 70)
print(f"\n  Razão observado/esperado na lua cheia: {vir(razao_v, 4)}  IC 95% [{vir(ic_v[0], 4)}; {vir(ic_v[1], 4)}]")
print(f"  Pelo modelo com a lua como variável: {vir(vag['razao_modelo'], 4)}  [{vir(vag['ic_modelo'][0], 4)}; {vir(vag['ic_modelo'][1], 4)}]")
print(f"  Em bebês: {vir((razao_v - 1) * d.vaginal.mean(), 0)} a mais num dia de {mil(d.vaginal.mean())} partos vaginais")
print(f"  Luas cheias acima do esperado: {acima_v} de {n_lc} · chance de a razão ser maior que 1: {vir(p_maior_v * 100, 0)}%")
print(f"  No calendário deslocado, o verdadeiro fica em {posicao_v}º lugar entre {len(razoes_desloc_v)}.")
print(f"  Cesáreas, para comparar: razão {vir(ces['razao_modelo'], 4)}  [{vir(ces['ic_modelo'][0], 4)}; {vir(ces['ic_modelo'][1], 4)}]")

# CAMADA 5 · O periodograma (método de Kuss & Kuehn)
# Série: log dos nascimentos sem a média de cada mês. O dia da semana fica dentro, de propósito,
# para o ciclo de 7 dias servir de régua. Potência em cada período candidato, de 2 a 60 dias.
serie = np.log(d.nascimentos.values)
serie = serie - pd.Series(serie).groupby(d.ano_mes.values).transform('mean').values
t = np.arange(len(serie))
def potencia(periodo):
    return abs(np.sum(serie * np.exp(-2j * np.pi * t / periodo))) ** 2 / len(serie)
periodos = np.linspace(2, 60, 4000)
P = np.array([potencia(p) for p in periodos])
P_lua = potencia(SINODICO); P_semana = potencia(7.0); P_meia_lua = potencia(SINODICO / 2)
percentil_lua = np.mean(P <= P_lua) * 100
print("\n" + "=" * 70)
print("  CAMADA 5 · O PERIODOGRAMA")
print("=" * 70)
print(f"\n  Potência no ciclo de 7 dias (semana):          {vir(P_semana, 2)}")
print(f"  Potência no ciclo de 29,53 dias (lua):         {vir(P_lua, 4)}  ({vir(P_semana / P_lua, 0)} vezes menor que a semana)")
print(f"  Potência no ciclo de 14,77 dias (meia lua):    {vir(P_meia_lua, 4)}")
print(f"  Entre {len(periodos)} períodos de 2 a 60 dias, a potência da lua está no percentil {vir(percentil_lua, 0)}.")
picos = [i for i in range(1, len(P) - 1) if P[i] > P[i - 1] and P[i] > P[i + 1]]
picos = sorted(picos, key=lambda i: -P[i])
escolhidos = []
for i in picos:
    if all(abs(periodos[i] - periodos[j]) > 0.5 for j in escolhidos):
        escolhidos.append(i)
picos = escolhidos[:3]
print(f"  Maiores picos: " + ", ".join(f"{vir(periodos[i], 1)} d" for i in picos))
print(f"  Kuss & Kuehn (2008), com 4,07 milhões de nascimentos alemães: p = 0,688 no período da lua.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · O calendário do hospital
fig1, ax1 = plt.subplots(figsize=(12, 8))
x = np.arange(7)
ef_vag = (np.exp(np.r_[0, vag['b'][:6]]) - 1) * 100
ef_ces = (np.exp(np.r_[0, ces['b'][:6]]) - 1) * 100
ef_tot = (efeito_dow - 1) * 100
ax1.bar(x - 0.27, ef_tot, width=0.26, color=CINZA, alpha=0.9, label='todos os nascimentos')
ax1.bar(x, ef_vag, width=0.26, color=VERDE, alpha=0.9, label='partos vaginais')
ax1.bar(x + 0.27, ef_ces, width=0.26, color=VERMELHO, alpha=0.9, label='cesáreas')
for i in range(7):
    if i > 0:
        ax1.text(x[i] + 0.27, ef_ces[i] - 1.5, f"{vir(ef_ces[i], 0)}%", ha='center', va='top', fontsize=11, color=VERMELHO)
        ax1.text(x[i], ef_vag[i] - 1.5, f"{vir(ef_vag[i], 0)}%", ha='center', va='top', fontsize=11, color=VERDE)
ax1.axhline((razao - 1) * 100, color=DOURADO, lw=2.5, ls='--')
ax1.text(6.45, (razao - 1) * 100 + 1, f"lua cheia: {vir((razao - 1) * 100)}%", ha='right', fontsize=12, color=DOURADO, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(['segunda', 'terça', 'quarta', 'quinta', 'sexta', 'sábado', 'domingo'], fontsize=12)
ax1.set_ylabel('Diferença em relação à segunda-feira (%)', fontsize=12)
ax1.set_ylim(-55, 8)
ax1.set_title('O calendário do hospital pesa; a lua, quase nada\n'
              f'domingo tem {vir(-ef_tot[6], 0)}% menos nascimentos que segunda; o dia de lua cheia tem {vir((razao - 1) * 100)}% a mais que o esperado',
              fontsize=14, pad=16)
ax1.legend(frameon=False, fontsize=12, loc='lower left')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-136-grafico-01-calendario.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · As 74 luas cheias
fig2, ax2 = plt.subplots(figsize=(12, 8))
cores = np.where(lc.razao > 1, DOURADO, CINZA)
ax2.scatter(lc.data, (lc.razao - 1) * 100, s=70, c=cores, zorder=3, edgecolor='white', lw=0.8)
ax2.axhline(0, color='#333', lw=1.2)
ax2.axhspan((ic_bs[0] - 1) * 100, (ic_bs[1] - 1) * 100, color=DOURADO, alpha=0.18, lw=0)
ax2.axhline((razao - 1) * 100, color=DOURADO, lw=2.5)
ax2.text(lc.data.iloc[-1] + pd.Timedelta(days=50), (razao - 1) * 100,
         f"média {vir((razao - 1) * 100)}%\nIC 95% [{vir((ic_bs[0] - 1) * 100)}; {vir((ic_bs[1] - 1) * 100)}]",
         va='center', fontsize=11, color='#6b5a1e')
ax2.set_xlim(lc.data.iloc[0] - pd.Timedelta(days=30), lc.data.iloc[-1] + pd.Timedelta(days=300))
ax2.set_ylabel('Nascimentos no dia de lua cheia, acima ou abaixo do esperado (%)', fontsize=12)
ax2.set_ylim(-9, 9)
ax2.set_title(f'As {n_lc} luas cheias, uma a uma\n'
              f'em {acima} delas nasceram mais bebês do que o esperado para o dia ({vir(acima / n_lc * 100, 0)}%); '
              f'em {n_lc - acima}, menos', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-136-grafico-02-as-luas-cheias.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · O calendário deslocado
fig3, ax3 = plt.subplots(figsize=(12, 8))
ks = np.array(list(DESLOC))
cores3 = [VERMELHO if k == 0 else CINZA for k in ks]
ax3.bar(ks, (razoes_desloc - 1) * 100, color=cores3, alpha=0.9, width=0.75)
ax3.axhline(0, color='#333', lw=1.2)
ax3.annotate(f"calendário verdadeiro\n{vir((razao - 1) * 100)}%, {posicao}º de {len(ks)}",
             xy=(0, (razao - 1) * 100), xytext=(4.5, 0.95), fontsize=12, color=VERMELHO, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.5))
ax3.set_xticks(ks[::3])
ax3.set_xticklabels([f"{k:+d}" if k else "0" for k in ks[::3]], fontsize=11)
ax3.set_xlabel('Deslocamento do calendário lunar (dias)', fontsize=12)
ax3.set_ylabel('Nascimentos nos "dias de lua cheia", acima ou abaixo do esperado (%)', fontsize=12)
ax3.set_ylim(-1.2, 1.2)
ax3.set_title('Trinta calendários, um verdadeiro\n'
              'a mesma conta feita com a lua cheia adiantada ou atrasada de 1 a 15 dias', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-136-grafico-03-calendario-deslocado.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · O periodograma
fig4, ax4 = plt.subplots(figsize=(12, 8))
ax4.plot(periodos, P, color=VERDE, lw=1.4)
ax4.set_yscale('log')
ax4.set_ylim(1e-4, 40)
ax4.axvline(SINODICO, color=DOURADO, lw=2, ls='--')
ax4.axvline(SINODICO / 2, color=DOURADO, lw=1.2, ls=':')
ax4.scatter([SINODICO], [P_lua], s=120, color=DOURADO, zorder=4, edgecolor='#333')
ax4.annotate(f"lua · 29,53 dias\npotência {vir(P_lua, 4)}\npercentil {vir(percentil_lua, 0)}",
             xy=(SINODICO, P_lua), xytext=(36, P_lua * 3), fontsize=12, color='#6b5a1e', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=DOURADO, lw=1.5))
ax4.annotate(f"semana · 7 dias\npotência {vir(P_semana, 1)}", xy=(7, P_semana), xytext=(11, P_semana * 0.6),
             fontsize=12, color=VERDE, fontweight='bold', arrowprops=dict(arrowstyle='->', color=VERDE, lw=1.5))
ax4.text(3.5, 3, "meia\nsemana", fontsize=10, color=VERDE, ha='center')
ax4.text(SINODICO / 2 + 0.5, 20, "meia lua\n14,77 dias", fontsize=10, color='#6b5a1e')
ax4.set_xlabel('Período do ciclo (dias)', fontsize=12)
ax4.set_ylabel('Potência do ciclo na série diária (escala log, cortada em 10⁻⁴)', fontsize=12)
ax4.set_title('A semana aparece; a lua, não\n'
              'o método de Kuss & Kuehn (2008) aplicado aos 15,7 milhões de nascimentos brasileiros', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-136-grafico-04-periodograma.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Em 44 das 74 luas cheias nasceram mais bebês do que o esperado para aquele dia. É 59%, com
intervalo de 48% a 70%. Uma moeda honesta dá 50%.**

O excesso, quando existe, é de 0,5%. São 35 bebês a mais num dia de 7.200, e a margem
de erro inclui o zero. O domingo tira 2.200. O feriado tira 1.100. Se a lua cheia faz
alguma coisa com a maternidade, ela faz sessenta vezes menos do que o dia da semana.

O sinal mais forte da série está nos partos vaginais, onde o hospital não escolhe a
data, 0,6% a mais, com intervalo de -0,04% a 1,3%. Ainda assim, entre trinta calendários
lunares deslocados, um deu resultado maior do que o verdadeiro. E o periodograma, que
procura qualquer ciclo escondido na série, encontra a semana com clareza e não encontra
a lua.

A conta brasileira chega aonde a alemã chegou com 4 milhões de nascimentos e a
americana com 564 mil. Quem já viu um plantão lotado numa noite de lua cheia viu de
verdade. Também viu, e não guardou, os plantões lotados nas outras noites.

*Qual crença sua sobreviveria a ser contada todos os dias, e não só nos dias marcantes?*

---

### ⚠️ Limitações do Modelo

- O dia de lua cheia é o dia civil de Brasília que contém o instante da lua cheia. O país
  tem quatro fusos e a lua cheia pode cair perto da meia-noite; a janela de três dias
  (razão 1,0025) é a conferência.
- O esperado de cada dia vem de um modelo com dia da semana, mês e feriados nacionais.
  Feriados estaduais e municipais, pontes e greves ficam no resíduo, que varia 3,5% de um
  dia para outro.
- O calendário deslocado situa o resultado entre trinta alternativas, e não é um teste
  formal. A ordem de 3º em 30 (e de 2º em 30 nos partos vaginais) é compatível com acaso
  e com um efeito pequeno demais para separar do ruído em seis anos.
- O SINASC registra a data de nascimento, e não a hora de início do trabalho de parto.
  Um efeito sobre o início do trabalho de parto se espalharia por dois dias civis.
- Seis anos de dados. O estudo alemão usou 38. A direção do resultado é a mesma.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*